# YOLOv26m-cls Loss Ablation - Original Pipeline Preserved

Runs only `yolo26m-cls` using the original dataset materialization, split, YOLO dataset preparation, preprocessing/augmentation inside Ultralytics training, evaluation, checkpoint, reporting, and plots. Adds three loss settings: baseline CrossEntropy, ASLSingleLabel, and LDAM.


In [1]:
print("Running YOLOv26m-cls loss ablation on Linux/RTX4090. Dataset will be materialized at ./uynnhy/processed-images and outputs at ./loss_ablation_yolo26m_original_pipeline_outputs.")


Running YOLOv26m-cls loss ablation on Linux/RTX4090. Dataset will be materialized at ./uynnhy/processed-images and outputs at ./loss_ablation_yolo26m_original_pipeline_outputs.


## 1. Install dependencies

In [2]:
import importlib.util
import subprocess
import sys

packages = [
    "kagglehub",
    "timm>=1.0.0",
    "torchmetrics",
    "ultralytics>=8.4.0",
    "onnx",
    "onnxscript",
    "ipywidgets",
]


def import_name_for(package_spec: str) -> str:
    package = package_spec.split(">=")[0].split("==")[0].split("<")[0]
    return {
        "ultralytics": "ultralytics",
        "torchmetrics": "torchmetrics",
        "onnxscript": "onnxscript",
        "kagglehub": "kagglehub",
        "ipywidgets": "ipywidgets",
    }.get(package, package.replace("-", "_"))


missing = [pkg for pkg in packages if importlib.util.find_spec(import_name_for(pkg)) is None]
if missing:
    print("Installing missing dependencies:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing])
else:
    print("All training dependencies are already available.")


Installing missing dependencies: ['ipywidgets']



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 2. Configuration, dataset download, and fixed split

In [3]:
import gc
import json
import os
import random
import shutil
import time
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as tv_models
import torchvision.transforms as transforms
from IPython.display import display
from PIL import Image
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
try:
    torch.use_deterministic_algorithms(False)
except Exception:
    pass
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "uynnhy" / "processed-images"
OUTPUT_DIR = PROJECT_ROOT / "loss_ablation_yolo26m_original_pipeline_outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
EXPORT_DIR = OUTPUT_DIR / "exports"
YOLO_DATA_DIR = OUTPUT_DIR / "yolo_dataset"
YOLO_RUNS_DIR = OUTPUT_DIR / "yolo_runs"
REPORT_DIR = OUTPUT_DIR / "reports"
PLOT_DIR = OUTPUT_DIR / "plots"
CONVERSION_ROOT = OUTPUT_DIR / "litert_conversions"

for directory in [CHECKPOINT_DIR, EXPORT_DIR, YOLO_DATA_DIR, YOLO_RUNS_DIR, REPORT_DIR, PLOT_DIR, CONVERSION_ROOT, DATASET_ROOT.parent]:
    directory.mkdir(parents=True, exist_ok=True)

CLASS_DIRS = ["1. Healthy", "2. BG", "3. WSSV", "4. WSSV_BG"]
CLASS_NAMES = ["Healthy", "BG", "WSSV", "WSSV_BG"]
CLASS_TO_IDX = {folder: idx for idx, folder in enumerate(CLASS_DIRS)}
NUM_CLASSES = len(CLASS_DIRS)


def has_class_folders(path: Path) -> bool:
    return all((path / class_dir).exists() for class_dir in CLASS_DIRS)


def find_processed_images_dir(root: Path) -> Path | None:
    if has_class_folders(root):
        return root
    for candidate in [root / "processed_images", root / "processed-images", root / "data", root / "dataset"]:
        if has_class_folders(candidate):
            return candidate
    if root.exists():
        for candidate in sorted([p for p in root.rglob("*") if p.is_dir()]):
            if has_class_folders(candidate):
                return candidate
    return None


def materialize_dataset() -> Path:
    existing = find_processed_images_dir(DATASET_ROOT)
    if existing is not None:
        print(f"Dataset already available at: {DATASET_ROOT}")
        return existing
    cache_path = Path(kagglehub.dataset_download("uynnhy/processed-images"))
    print(f"KaggleHub cache path: {cache_path}")
    if DATASET_ROOT.exists() and not any(DATASET_ROOT.iterdir()):
        shutil.rmtree(DATASET_ROOT)
    if not DATASET_ROOT.exists():
        shutil.copytree(cache_path, DATASET_ROOT, dirs_exist_ok=True)
    else:
        shutil.copytree(cache_path, DATASET_ROOT, dirs_exist_ok=True)
    resolved = find_processed_images_dir(DATASET_ROOT)
    if resolved is None:
        raise RuntimeError(f"Could not find class folders under {DATASET_ROOT}")
    return resolved


IMG_SIZE = 224
EPOCHS = 30
PATIENCE = 15
PAPER_BATCH_SIZE = 128
MICRO_BATCH_SIZE = 32
ACCUMULATION_STEPS = max(1, PAPER_BATCH_SIZE // MICRO_BATCH_SIZE)
EVAL_BATCH_SIZE = PAPER_BATCH_SIZE
YOLO_BATCH_SIZE = PAPER_BATCH_SIZE
WARMUP_EPOCHS = 5
WARMUP_HEAD_LR = 1e-3
BACKBONE_FINETUNE_LR = 2e-5
HEAD_FINETUNE_LR = 1e-4
STEP_SIZE = 3
STEP_GAMMA = 0.9
NUM_WORKERS = min(8, max(2, (os.cpu_count() or 4) // 2))
ENABLE_GPU_LOGGING = True
GPU_LOG_EVERY_N_EPOCHS = 1
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16
PIN_MEMORY = torch.cuda.is_available()
PERSISTENT_WORKERS = NUM_WORKERS > 0

DATA_DIR = materialize_dataset()

TIMM_MODELS = []
YOLO_MODELS = ["yolo26m-cls"]
ALL_MODELS = YOLO_MODELS
LOSS_RUNS = ["baseline_ce", "asl", "ldam"]

LOSS_LABELS = {
    "baseline_ce": "Baseline CE",
    "asl": "ASL",
    "ldam": "LDAM",
}

LOSS_PAPERS = {
    "baseline_ce": "Default CrossEntropyLoss",
    "asl": "Asymmetric Loss For Multi-Label Classification",
    "ldam": "Learning Imbalanced Datasets with Label-Distribution-Aware Margin Loss",
}

RUN_EXPORT_AFTER_TRAINING = False


def gpu_status_text() -> str:
    if not torch.cuda.is_available():
        return "CUDA unavailable"
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    max_allocated = torch.cuda.max_memory_allocated() / 1024**2
    text = f"torch CUDA memory allocated/reserved/max: {allocated:.1f}/{reserved:.1f}/{max_allocated:.1f} MB"
    try:
        completed = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=5,
        )
        if completed.returncode == 0 and completed.stdout.strip():
            util, mem_used, mem_total, power, temp = [part.strip() for part in completed.stdout.strip().split(",")]
            text += f" | nvidia-smi util={util}% mem={mem_used}/{mem_total} MB power={power} W temp={temp} C"
        else:
            text += f" | nvidia-smi failed: {completed.stderr.strip()[-200:]}"
    except Exception as exc:
        text += f" | nvidia-smi unavailable: {type(exc).__name__}: {exc}"
    return text


def print_gpu_status(label: str):
    if ENABLE_GPU_LOGGING:
        print(f"[GPU] {label}: {gpu_status_text()}")


print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"NUM_WORKERS: {NUM_WORKERS}")
print_gpu_status("startup")


/opt/miniconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


Using device: cuda
GPU: NVIDIA GeForce RTX 4090
Dataset already available at: /home/drnguyenvinh/notebooks/uynnhy/processed-images
Project root: /home/drnguyenvinh/notebooks
Dataset root: /home/drnguyenvinh/notebooks/uynnhy/processed-images
Data directory: /home/drnguyenvinh/notebooks/uynnhy/processed-images/processed_images
Output directory: /home/drnguyenvinh/notebooks/loss_ablation_yolo26m_original_pipeline_outputs
NUM_WORKERS: 8
[GPU] startup: torch CUDA memory allocated/reserved/max: 0.0/0.0/0.0 MB | nvidia-smi util=29% mem=1226/24564 MB power=13.12 W temp=53 C


In [4]:
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)


train_loader_generator = torch.Generator()
train_loader_generator.manual_seed(SEED)
eval_loader_generator = torch.Generator()
eval_loader_generator.manual_seed(SEED)


def discover_processed_images(data_dir: Path) -> pd.DataFrame:
    rows = []
    for class_dir in CLASS_DIRS:
        folder = data_dir / class_dir
        if not folder.exists():
            print(f"Warning: missing class folder: {folder}")
            continue
        for path in sorted(folder.iterdir()):
            if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                rows.append({"path": str(path), "class_dir": class_dir, "label": CLASS_TO_IDX[class_dir]})
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError(f"No images found under {data_dir}.")
    return frame


df = discover_processed_images(DATA_DIR)
print(f"Loaded {len(df)} processed images from {DATA_DIR}")
display(df["class_dir"].value_counts().reindex(CLASS_DIRS).rename("count").to_frame())

train_df, tmp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED,
    shuffle=True,
)
val_df, test_df = train_test_split(
    tmp_df,
    test_size=0.50,
    stratify=tmp_df["label"],
    random_state=SEED,
    shuffle=True,
)

CLASS_COUNTS = train_df["label"].value_counts().sort_index().reindex(range(NUM_CLASSES), fill_value=1).astype(int).tolist()
print(f"Train class counts: {dict(zip(CLASS_NAMES, CLASS_COUNTS))}")

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{split_name}: {len(split_df)} images")
    print(split_df["class_dir"].value_counts().reindex(CLASS_DIRS).to_dict())

assert set(train_df["path"]).isdisjoint(set(val_df["path"]))
assert set(train_df["path"]).isdisjoint(set(test_df["path"]))
assert set(val_df["path"]).isdisjoint(set(test_df["path"]))
print("Image-level split overlap check passed.")


Loaded 1149 processed images from /home/drnguyenvinh/notebooks/uynnhy/processed-images/processed_images


,count
class_dir,
1. Healthy,403
2. BG,198
3. WSSV,328
4. WSSV_BG,220


Train class counts: {'Healthy': 282, 'BG': 139, 'WSSV': 229, 'WSSV_BG': 154}
train: 804 images
{'1. Healthy': 282, '2. BG': 139, '3. WSSV': 229, '4. WSSV_BG': 154}
val: 172 images
{'1. Healthy': 60, '2. BG': 30, '3. WSSV': 49, '4. WSSV_BG': 33}
test: 173 images
{'1. Healthy': 61, '2. BG': 29, '3. WSSV': 50, '4. WSSV_BG': 33}
Image-level split overlap check passed.


## 3. Original TIMM preprocessing, augmentation, and dataloaders

In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.82, 1.0),
        ratio=(0.90, 1.10),
        interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10, interpolation=InterpolationMode.BICUBIC),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(236, interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class ShrimpFrameDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"])


def make_loader(dataset, batch_size, shuffle, generator):
    kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
        generator=generator,
    )
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = PERSISTENT_WORKERS
        kwargs["prefetch_factor"] = 4
    return DataLoader(**kwargs)


train_loader = make_loader(
    ShrimpFrameDataset(train_df, train_transform),
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    generator=train_loader_generator,
)
val_loader = make_loader(
    ShrimpFrameDataset(val_df, eval_transform),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    generator=eval_loader_generator,
)
test_loader = make_loader(
    ShrimpFrameDataset(test_df, eval_transform),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    generator=eval_loader_generator,
)


## 4. Loss functions

In [6]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8, disable_torch_grad_focal_loss=True):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.disable_torch_grad_focal_loss = disable_torch_grad_focal_loss
        self.eps = eps

    def forward(self, x, y):
        x_sigmoid = torch.sigmoid(x)
        xs_pos = x_sigmoid
        xs_neg = 1 - x_sigmoid
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)
        los_pos = y * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1 - y) * torch.log(xs_neg.clamp(min=self.eps))
        loss = los_pos + los_neg
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            grad_enabled = torch.is_grad_enabled()
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(False)
            pt0 = xs_pos * y
            pt1 = xs_neg * (1 - y)
            pt = pt0 + pt1
            one_sided_gamma = self.gamma_pos * y + self.gamma_neg * (1 - y)
            one_sided_w = torch.pow(1 - pt, one_sided_gamma)
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(grad_enabled)
            loss *= one_sided_w
        return -loss.sum()


class AsymmetricLossOptimized(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8, disable_torch_grad_focal_loss=False):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.disable_torch_grad_focal_loss = disable_torch_grad_focal_loss
        self.eps = eps
        self.targets = self.anti_targets = self.xs_pos = self.xs_neg = self.asymmetric_w = self.loss = None

    def forward(self, x, y):
        self.targets = y
        self.anti_targets = 1 - y
        self.xs_pos = torch.sigmoid(x)
        self.xs_neg = 1.0 - self.xs_pos
        if self.clip is not None and self.clip > 0:
            self.xs_neg.add_(self.clip).clamp_(max=1)
        self.loss = self.targets * torch.log(self.xs_pos.clamp(min=self.eps))
        self.loss.add_(self.anti_targets * torch.log(self.xs_neg.clamp(min=self.eps)))
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            grad_enabled = torch.is_grad_enabled()
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(False)
            self.xs_pos = self.xs_pos * self.targets
            self.xs_neg = self.xs_neg * self.anti_targets
            self.asymmetric_w = torch.pow(
                1 - self.xs_pos - self.xs_neg,
                self.gamma_pos * self.targets + self.gamma_neg * self.anti_targets,
            )
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(grad_enabled)
            self.loss *= self.asymmetric_w
        return -self.loss.sum()


class ASLSingleLabel(nn.Module):
    def __init__(self, gamma_pos=0, gamma_neg=4, eps=0.1, reduction="mean"):
        super().__init__()
        self.eps = eps
        self.logsoftmax = nn.LogSoftmax(dim=-1)
        self.targets_classes = []
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.reduction = reduction

    def forward(self, inputs, target):
        target = target.long().view(-1)
        num_classes = inputs.size()[-1]
        log_preds = self.logsoftmax(inputs)
        self.targets_classes = torch.zeros_like(inputs).scatter_(1, target.unsqueeze(1), 1)
        targets = self.targets_classes
        anti_targets = 1 - targets
        xs_pos = torch.exp(log_preds)
        xs_neg = 1 - xs_pos
        xs_pos = xs_pos * targets
        xs_neg = xs_neg * anti_targets
        asymmetric_w = torch.pow(1 - xs_pos - xs_neg, self.gamma_pos * targets + self.gamma_neg * anti_targets)
        log_preds = log_preds * asymmetric_w
        if self.eps > 0:
            self.targets_classes = self.targets_classes.mul(1 - self.eps).add(self.eps / num_classes)
        loss = -self.targets_classes.mul(log_preds)
        loss = loss.sum(dim=-1)
        if self.reduction == "mean":
            loss = loss.mean()
        elif self.reduction == "sum":
            loss = loss.sum()
        return loss


def focal_loss(input_values, gamma):
    p = torch.exp(-input_values)
    loss = (1 - p) ** gamma * input_values
    return loss.mean()


class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=0.0):
        super().__init__()
        assert gamma >= 0
        self.gamma = gamma
        self.weight = weight

    def forward(self, input, target):
        return focal_loss(F.cross_entropy(input, target, reduction="none", weight=self.weight), self.gamma)


class LDAMLoss(nn.Module):
    def __init__(self, cls_num_list, max_m=0.5, weight=None, s=30):
        super().__init__()
        cls_num_list = np.asarray(cls_num_list, dtype=np.float64)
        cls_num_list = np.maximum(cls_num_list, 1.0)
        m_list = 1.0 / np.sqrt(np.sqrt(cls_num_list))
        m_list = m_list * (max_m / np.max(m_list))
        self.register_buffer("m_list", torch.tensor(m_list, dtype=torch.float32))
        assert s > 0
        self.s = s
        if weight is None:
            self.weight = None
        else:
            self.register_buffer("weight", torch.tensor(weight, dtype=torch.float32))

    def forward(self, x, target):
        target = target.long().view(-1)
        index = torch.zeros_like(x, dtype=torch.bool)
        index.scatter_(1, target.view(-1, 1), True)
        batch_m = self.m_list[target].view((-1, 1)).to(x.device)
        x_m = x - batch_m
        output = torch.where(index, x_m, x)
        weight = self.weight.to(x.device) if self.weight is not None else None
        return F.cross_entropy(self.s * output, target, weight=weight)


def make_loss(loss_name):
    if loss_name == "baseline_ce":
        return nn.CrossEntropyLoss()
    if loss_name == "asl":
        return ASLSingleLabel(gamma_pos=0, gamma_neg=4, eps=0.1, reduction="mean")
    if loss_name == "ldam":
        return LDAMLoss(CLASS_COUNTS, max_m=0.5, s=30)
    raise ValueError(loss_name)


## 5. Original model helpers with loss-ablation support

In [7]:
TIMM_ALIASES = {
    "convnext_tiny_in22k": ["convnext_tiny.fb_in22k", "convnext_tiny.fb_in1k", "convnext_tiny"],
    "mobilenet_v3_large": ["mobilenetv3_large_100.ra_in1k", "mobilenetv3_large_100", "tf_mobilenetv3_large_100"],
    "efficientnet_b0": ["efficientnet_b0.ra_in1k", "tf_efficientnet_b0.ns_jft_in1k", "efficientnet_b0"],
    "repvgg_a0": ["repvgg_a0.rvgg_in1k", "repvgg_a0"],
    "efficientnet_v2_s": ["tf_efficientnetv2_s.in21k_ft_in1k", "tf_efficientnetv2_s", "efficientnetv2_rw_s.ra2_in1k"],
    "shufflenet_v2_x1_0": ["shufflenet_v2_x1_0"],
    "fastvit_t8": ["fastvit_t8.apple_in1k", "fastvit_t8"],
    "edgenext_xx_small": ["edgenext_xx_small.in1k", "edgenext_xx_small"],
    "mobileone_s0": ["mobileone_s0.apple_in1k", "mobileone_s0"],
    "mobilevit_s": ["mobilevit_s.cvnets_in1k", "mobilevit_s"],
    "mobilenetv4_conv_small": ["mobilenetv4_conv_small.e2400_r224_in1k", "mobilenetv4_conv_small"],
    "mnasnet_100": ["mnasnet_100.rmsp_in1k", "mnasnet_100"],
    "ghostnetv2_100": ["ghostnetv2_100.in1k", "ghostnetv2_100"],
    "rexnet_100": ["rexnet_100.nav_in1k", "rexnet_100"],
    "squeezenet1_1": ["squeezenet1_1"],
    "mobilenetv4_hybrid_medium": ["mobilenetv4_hybrid_medium.e200_r256_in12k_ft_in1k", "mobilenetv4_hybrid_medium"],
    "efficientvit_m1": ["efficientvit_m1.r224_in1k", "efficientvit_m1"],
}


def sanitize_name(name: str) -> str:
    return name.replace("/", "_").replace(" ", "_").replace(".", "_")


def count_params(model) -> float:
    return sum(param.numel() for param in model.parameters()) / 1e6


def resolve_timm_name(display_name: str) -> str:
    candidates = TIMM_ALIASES.get(display_name, [display_name])
    available = set(timm.list_models(pretrained=False))
    for candidate in candidates:
        if candidate in available:
            return candidate
    pattern_hits = []
    for candidate in candidates:
        pattern_hits.extend(timm.list_models(candidate + "*", pretrained=False))
    if pattern_hits:
        return sorted(pattern_hits)[0]
    raise ValueError(f"No TIMM model found for {display_name}. Tried: {candidates}")


class TimmShrimpXNet(nn.Module):
    def __init__(self, timm_name: str, pretrained: bool):
        super().__init__()
        try:
            self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0, global_pool="avg")
        except TypeError:
            self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0)
        was_training = self.backbone.training
        self.backbone.eval()
        with torch.no_grad():
            sample = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
            features = self.backbone(sample)
            if isinstance(features, (list, tuple)):
                features = features[-1]
            num_features = features.flatten(1).shape[1]
        self.backbone.train(was_training)
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(512, NUM_CLASSES),
        )

    def forward(self, x):
        x = self.backbone(x)
        if isinstance(x, (list, tuple)):
            x = x[-1]
        x = torch.flatten(x, 1)
        return self.classifier(x)


class TorchvisionShrimpXNet(nn.Module):
    def __init__(self, model_name: str, pretrained: bool):
        super().__init__()
        if model_name == "shufflenet_v2_x1_0":
            weights = tv_models.ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1 if pretrained else None
            backbone = tv_models.shufflenet_v2_x1_0(weights=weights)
            num_features = backbone.fc.in_features
            backbone.fc = nn.Identity()
        elif model_name == "squeezenet1_1":
            weights = tv_models.SqueezeNet1_1_Weights.IMAGENET1K_V1 if pretrained else None
            backbone = tv_models.squeezenet1_1(weights=weights)
            num_features = 512
            backbone.classifier = nn.Sequential(
                nn.Dropout(p=0.5),
                nn.Conv2d(512, num_features, kernel_size=1),
                nn.ReLU(inplace=True),
                nn.AdaptiveAvgPool2d((1, 1)),
            )
        else:
            raise ValueError(f"Unsupported torchvision fallback model: {model_name}")
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(512, NUM_CLASSES),
        )

    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def create_timm_classifier(display_name: str):
    if display_name in {"shufflenet_v2_x1_0", "squeezenet1_1"}:
        try:
            return TorchvisionShrimpXNet(display_name, pretrained=True).to(device), display_name, True
        except Exception as pretrained_error:
            print(f"Pretrained torchvision weights failed for {display_name}: {type(pretrained_error).__name__}: {pretrained_error}")
            return TorchvisionShrimpXNet(display_name, pretrained=False).to(device), display_name, False
    timm_name = resolve_timm_name(display_name)
    try:
        model = TimmShrimpXNet(timm_name, pretrained=True)
        pretrained = True
    except Exception as pretrained_error:
        print(f"Pretrained weights failed for {display_name} ({timm_name}): {type(pretrained_error).__name__}: {pretrained_error}")
        model = TimmShrimpXNet(timm_name, pretrained=False)
        pretrained = False
    return model.to(device), timm_name, pretrained


def classifier_parameters(model):
    if hasattr(model, "classifier"):
        return list(model.classifier.parameters())
    params = []
    classifier = model.get_classifier() if hasattr(model, "get_classifier") else None
    if isinstance(classifier, nn.Module):
        params = list(classifier.parameters())
    elif isinstance(classifier, (list, tuple, nn.ModuleList)):
        for module in classifier:
            if isinstance(module, nn.Module):
                params.extend(list(module.parameters()))
    if not params:
        head_tokens = ("classifier", "head", "fc")
        params = [param for name, param in model.named_parameters() if any(token in name.lower() for token in head_tokens)]
    if not params:
        raise RuntimeError("Could not identify classifier/head parameters for warmup.")
    return params


def freeze_backbone_for_warmup(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in classifier_parameters(model):
        param.requires_grad = True


def make_warmup_optimizer(model):
    return optim.Adam([param for param in model.parameters() if param.requires_grad], lr=WARMUP_HEAD_LR)


def unfreeze_module(module):
    for param in module.parameters():
        param.requires_grad = True


def unfreeze_final_backbone_portion(model, display_name: str):
    backbone = model.backbone if hasattr(model, "backbone") else model
    trainable_modules = []
    for param in model.parameters():
        param.requires_grad = False
    for param in classifier_parameters(model):
        param.requires_grad = True
    if hasattr(backbone, "features") and isinstance(backbone.features, (nn.Sequential, nn.ModuleList, list, tuple)):
        features = backbone.features
        if "convnext" in display_name.lower() and len(features) > 5:
            selected = list(features[5:])
        else:
            start = max(0, len(features) - max(1, len(features) // 3))
            selected = list(features[start:])
        trainable_modules.extend(selected)
    elif hasattr(backbone, "stages") and isinstance(backbone.stages, (nn.Sequential, nn.ModuleList, list, tuple)):
        stages = backbone.stages
        start = max(0, len(stages) - max(1, len(stages) // 3))
        trainable_modules.extend(list(stages[start:]))
    elif hasattr(backbone, "blocks") and isinstance(backbone.blocks, (nn.Sequential, nn.ModuleList, list, tuple)):
        blocks = backbone.blocks
        start = max(0, len(blocks) - max(1, len(blocks) // 3))
        trainable_modules.extend(list(blocks[start:]))
    else:
        excluded = {"classifier", "head", "fc", "global_pool", "pool", "avgpool"}
        children = [child for name, child in backbone.named_children() if name not in excluded and not name.startswith("head")]
        trainable_modules.extend(children[-2:] if len(children) >= 2 else children)
    for attr in ["norm", "norm_head", "head_norm", "pre_head", "final_conv"]:
        module = getattr(backbone, attr, None)
        if isinstance(module, nn.Module):
            trainable_modules.append(module)
    for module in trainable_modules:
        unfreeze_module(module)
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def make_finetune_optimizer(model):
    head_param_ids = {id(param) for param in classifier_parameters(model)}
    backbone_params = []
    head_params = []
    for param in model.parameters():
        if not param.requires_grad:
            continue
        if id(param) in head_param_ids:
            head_params.append(param)
        else:
            backbone_params.append(param)
    param_groups = []
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": BACKBONE_FINETUNE_LR})
    if head_params:
        param_groups.append({"params": head_params, "lr": HEAD_FINETUNE_LR})
    return optim.Adam(param_groups)


def extract_logits(output):
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, (list, tuple)):
        tensors = [item for item in output if isinstance(item, torch.Tensor)]
        if tensors:
            return tensors[-1]
    if hasattr(output, "logits"):
        return output.logits
    raise TypeError(f"Unsupported model output type: {type(output)}")


def forward_with_amp(model, ims):
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        return extract_logits(model(ims))


def predict_pytorch(model, loader, criterion=None, timed=False):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    if timed:
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
        with torch.no_grad():
            for _ in range(5):
                model(dummy)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        start = time.time()
    else:
        start = None
    with torch.no_grad():
        for ims, gts in loader:
            ims = ims.to(device, non_blocking=True)
            gts = gts.to(device, non_blocking=True)
            logits = forward_with_amp(model, ims)
            if criterion is not None:
                total_loss += criterion(logits.float(), gts).item() * ims.size(0)
            preds = torch.argmax(logits, dim=1)
            all_labels.extend(gts.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start if timed else None
    return {
        "loss": total_loss / max(1, len(all_labels)) if criterion is not None else None,
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall": recall_score(all_labels, all_preds, average="macro", zero_division=0),
        "macro_f1": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "cohen_kappa": cohen_kappa_score(all_labels, all_preds),
        "elapsed": elapsed,
        "labels": all_labels,
        "preds": all_preds,
    }


def save_confusion_matrix(labels, preds, title, path):
    matrix = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(matrix)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, int(matrix[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 6. TIMM training with the original warmup/fine-tuning pipeline

In [8]:
def new_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def train_timm_model(model_name: str, loss_name: str) -> tuple[dict, list[dict]]:
    loss_label = LOSS_LABELS[loss_name]
    run_name = sanitize_name(f"{model_name}_{loss_name}")
    print("\n" + "=" * 90)
    print(f"Training TIMM classifier: {model_name} | {loss_label}")
    print("=" * 90)
    model, timm_name, pretrained = create_timm_classifier(model_name)
    print(f"Resolved TIMM model: {timm_name} | pretrained={pretrained}")
    print_gpu_status(f"{model_name} {loss_label} after model.to(device)")
    criterion = make_loss(loss_name).to(device)
    best_path = CHECKPOINT_DIR / f"best_{run_name}.pth"
    best_val_loss = float("inf")
    best_val_f1 = -1.0
    best_epoch = 0
    epochs_no_improve = 0
    history = []
    train_start = time.time()
    freeze_backbone_for_warmup(model)
    print(f"Warmup trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    optimizer = make_warmup_optimizer(model)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=STEP_GAMMA)
    scaler = new_grad_scaler()
    for epoch in range(EPOCHS):
        if epoch == WARMUP_EPOCHS:
            print("\n--- Switching to fine-tuning phase: unfreezing backbone with lower LR ---")
            trainable_count = unfreeze_final_backbone_portion(model, model_name)
            print(f"Fine-tune trainable parameters: {trainable_count:,}")
            optimizer = make_finetune_optimizer(model)
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=STEP_GAMMA)
            epochs_no_improve = 0
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        current_lrs = [group["lr"] for group in optimizer.param_groups]
        optimizer.zero_grad(set_to_none=True)
        for step, (ims, gts) in enumerate(tqdm(train_loader, desc=f"{model_name} {loss_label} epoch {epoch + 1}/{EPOCHS}", leave=False)):
            ims = ims.to(device, non_blocking=True)
            gts = gts.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = extract_logits(model(ims))
                loss = criterion(logits.float(), gts)
                scaled_loss = loss / ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
            if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            train_loss += loss.item() * ims.size(0)
            train_correct += (torch.argmax(logits, dim=1) == gts).sum().item()
            train_total += gts.size(0)
        scheduler.step()
        val_metrics = predict_pytorch(model, val_loader, criterion=criterion, timed=False)
        if (epoch + 1) % GPU_LOG_EVERY_N_EPOCHS == 0:
            print_gpu_status(f"{model_name} {loss_label} epoch {epoch + 1} end")
        train_acc = train_correct / max(1, train_total)
        phase = "warmup" if epoch < WARMUP_EPOCHS else "finetune"
        lr_text = ",".join(f"{lr:.2e}" for lr in current_lrs)
        epoch_record = {
            "Model": model_name,
            "Backend": "timm",
            "Loss": loss_label,
            "Loss Key": loss_name,
            "Epoch": epoch + 1,
            "Phase": phase,
            "LR": lr_text,
            "Train Loss": train_loss / max(1, train_total),
            "Train Accuracy": train_acc,
            "Val Loss": val_metrics["loss"],
            "Val Accuracy": val_metrics["accuracy"],
            "Val Precision": val_metrics["precision"],
            "Val Recall": val_metrics["recall"],
            "Val F1-Score": val_metrics["macro_f1"],
            "Val Cohen Kappa": val_metrics["cohen_kappa"],
        }
        history.append(epoch_record)
        pd.DataFrame(history).to_csv(REPORT_DIR / f"history_{run_name}.csv", index=False)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | Phase: {phase} | LR: {lr_text} | "
            f"Train Loss: {epoch_record['Train Loss']:.4f} - Acc: {train_acc:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} - Acc: {val_metrics['accuracy']:.4f} - Macro F1: {val_metrics['macro_f1']:.4f}"
        )
        improved = val_metrics["loss"] < best_val_loss
        if improved:
            best_val_loss = val_metrics["loss"]
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch + 1
            epochs_no_improve = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "display_name": model_name,
                    "timm_name": timm_name,
                    "backend_source": "torchvision" if model_name in {"shufflenet_v2_x1_0", "squeezenet1_1"} else "timm",
                    "pretrained": pretrained,
                    "loss": loss_name,
                    "loss_label": loss_label,
                    "num_classes": NUM_CLASSES,
                    "img_size": IMG_SIZE,
                    "class_names": CLASS_NAMES,
                },
                best_path,
            )
            print(f"  --> Saved best checkpoint: val loss {best_val_loss:.4f}, macro F1 {best_val_f1:.4f}")
        else:
            epochs_no_improve += 1
            print(f"  --> No improvement ({epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print("  --> Early stopping triggered.")
            break
    train_time = time.time() - train_start
    checkpoint = torch.load(best_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    test_metrics = predict_pytorch(model, test_loader, criterion=None, timed=True)
    inf_time = max(test_metrics["elapsed"], 1e-9)
    params_m = count_params(model)
    cm_path = PLOT_DIR / f"confusion_matrix_{run_name}.png"
    save_confusion_matrix(test_metrics["labels"], test_metrics["preds"], f"{model_name} | {loss_label}", cm_path)
    result = {
        "Model": model_name,
        "Backend": "timm",
        "Backend Name": timm_name,
        "Loss": loss_label,
        "Loss Key": loss_name,
        "Loss Paper": LOSS_PAPERS[loss_name],
        "Parameters (M)": round(params_m, 2),
        "Training Time (s)": round(train_time, 1),
        "Best Epoch": best_epoch,
        "Val F1-Score": round(best_val_f1, 4),
        "Best Val Loss": round(best_val_loss, 4),
        "Test Accuracy": round(test_metrics["accuracy"], 4),
        "Test Precision": round(test_metrics["precision"], 4),
        "Test Recall": round(test_metrics["recall"], 4),
        "Test F1-Score": round(test_metrics["macro_f1"], 4),
        "Cohen Kappa": round(test_metrics["cohen_kappa"], 4),
        "Inference Time (s)": round(inf_time, 2),
        "FPS": round(len(test_df) / inf_time, 1),
        "Latency (ms)": round((inf_time / len(test_df)) * 1000, 2),
        "Checkpoint Path": str(best_path),
        "Confusion Matrix Path": str(cm_path),
        "Export Format": "",
        "Export Path": "",
        "Export Note": "",
    }
    del model, optimizer, scheduler, criterion, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return result, history


## 7. Original YOLO classification pipeline with custom loss trainer

In [9]:
def prepare_yolo_dataset():
    if YOLO_DATA_DIR.exists():
        shutil.rmtree(YOLO_DATA_DIR)
    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        for class_dir in CLASS_DIRS:
            (YOLO_DATA_DIR / split_name / class_dir).mkdir(parents=True, exist_ok=True)
        for _, row in split_df.iterrows():
            src = Path(row["path"])
            dst = YOLO_DATA_DIR / split_name / row["class_dir"] / src.name
            if dst.exists():
                dst = dst.with_name(f"{dst.stem}_{abs(hash(str(src))) % 10_000_000}{dst.suffix}")
            shutil.copy2(src, dst)
    print(f"Prepared YOLO classification dataset at {YOLO_DATA_DIR}")


prepare_yolo_dataset()


def yolo_device_arg():
    return 0 if torch.cuda.is_available() else "cpu"


try:
    from ultralytics.models.yolo.classify.train import ClassificationTrainer
except Exception:
    from ultralytics.models.yolo.classify import ClassificationTrainer

from ultralytics.nn.tasks import ClassificationModel


ACTIVE_YOLO_LOSS_NAME = "baseline_ce"
ACTIVE_YOLO_CLASS_COUNTS = CLASS_COUNTS


class LossAblationClassificationLoss(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.loss_name = getattr(model, "loss_name", ACTIVE_YOLO_LOSS_NAME)
        self.loss_fcn = make_loss(self.loss_name)

    def forward(self, preds, batch):
        preds = preds[1] if isinstance(preds, (list, tuple)) else preds
        targets = batch["cls"].long().view(-1).to(preds.device)
        if isinstance(self.loss_fcn, nn.Module):
            self.loss_fcn = self.loss_fcn.to(preds.device)
        loss = self.loss_fcn(preds.float(), targets)
        loss_items = loss.detach()
        return loss, loss_items


class LossAblationClassificationModel(ClassificationModel):
    def init_criterion(self):
        return LossAblationClassificationLoss(self)


class LossAblationClassificationTrainer(ClassificationTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        nc = self.data["nc"] if isinstance(self.data, dict) and "nc" in self.data else NUM_CLASSES
        try:
            model = LossAblationClassificationModel(cfg, nc=nc, verbose=verbose)
        except TypeError:
            model = LossAblationClassificationModel(cfg, ch=3, nc=nc, verbose=verbose)
        model.loss_name = ACTIVE_YOLO_LOSS_NAME
        model.cls_num_list = ACTIVE_YOLO_CLASS_COUNTS
        if weights:
            model.load(weights)
        return model


def evaluate_yolo_model(yolo_model, eval_df: pd.DataFrame, timed=False):
    names = yolo_model.names
    name_to_idx = {value: int(key) for key, value in names.items()}
    source_paths = eval_df["path"].tolist()
    if timed:
        _ = yolo_model.predict(source=source_paths[:1], imgsz=IMG_SIZE, device=yolo_device_arg(), verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
    else:
        start = None
    preds = yolo_model.predict(
        source=source_paths,
        imgsz=IMG_SIZE,
        batch=YOLO_BATCH_SIZE,
        device=yolo_device_arg(),
        verbose=False,
    )
    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start if timed else None
    y_true = [name_to_idx[class_dir] for class_dir in eval_df["class_dir"].tolist()]
    y_pred = [int(result.probs.top1) for result in preds]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "elapsed": elapsed,
        "labels": y_true,
        "preds": y_pred,
    }


def read_yolo_history(run_dir: Path, model_name: str, loss_name: str):
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        hits = sorted(run_dir.glob("**/results.csv"))
        if not hits:
            return []
        results_csv = hits[-1]
    frame = pd.read_csv(results_csv)
    frame.columns = [col.strip() for col in frame.columns]
    history = []
    for idx, row in frame.iterrows():
        record = {
            "Model": model_name,
            "Backend": "yolo",
            "Loss": LOSS_LABELS[loss_name],
            "Loss Key": loss_name,
            "Epoch": int(row.get("epoch", idx + 1)),
        }
        for col in frame.columns:
            value = row[col]
            if pd.api.types.is_number(value):
                record[col] = value
        history.append(record)
    return history


def get_yolo_best_info(run_dir: Path):
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        hits = sorted(run_dir.glob("**/results.csv"))
        if not hits:
            return np.nan, np.nan
        results_csv = hits[-1]
    frame = pd.read_csv(results_csv)
    frame.columns = [col.strip() for col in frame.columns]
    metric_candidates = [
        "metrics/accuracy_top1",
        "metrics/accuracy_top5",
        "top1_acc",
        "accuracy_top1",
    ]
    metric_col = next((col for col in metric_candidates if col in frame.columns), None)
    if metric_col is None:
        metric_col = next((col for col in frame.columns if "top1" in col.lower()), None)
    if metric_col is None or frame.empty:
        return np.nan, np.nan
    idx = frame[metric_col].astype(float).idxmax()
    epoch = int(frame.loc[idx, "epoch"]) if "epoch" in frame.columns else int(idx) + 1
    value = float(frame.loc[idx, metric_col])
    return epoch, value


def train_yolo_model(model_name: str, loss_name: str) -> tuple[dict, list[dict]]:
    global ACTIVE_YOLO_LOSS_NAME, ACTIVE_YOLO_CLASS_COUNTS
    ACTIVE_YOLO_LOSS_NAME = loss_name
    ACTIVE_YOLO_CLASS_COUNTS = CLASS_COUNTS
    loss_label = LOSS_LABELS[loss_name]
    run_name = sanitize_name(f"{model_name}_{loss_name}")
    print("\n" + "=" * 90)
    print(f"Training YOLO classifier: {model_name} | {loss_label}")
    print("=" * 90)
    weights_name = f"{model_name}.pt"
    train_start = time.time()
    yolo = YOLO(weights_name)
    print_gpu_status(f"{model_name} {loss_label} before YOLO train")
    train_kwargs = dict(
        data=str(YOLO_DATA_DIR),
        task="classify",
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=YOLO_BATCH_SIZE,
        patience=PATIENCE,
        seed=SEED,
        project=str(YOLO_RUNS_DIR),
        name=run_name,
        exist_ok=True,
        device=yolo_device_arg(),
        verbose=True,
        workers=NUM_WORKERS,
        amp=USE_AMP,
        optimizer="AdamW",
        lr0=1.25e-3,
        lrf=0.01,
        cos_lr=True,
        cache=True,
    )
    if loss_name == "baseline_ce":
        yolo.train(**train_kwargs)
    else:
        yolo.train(trainer=LossAblationClassificationTrainer, **train_kwargs)
    print_gpu_status(f"{model_name} {loss_label} after YOLO train")
    train_time = time.time() - train_start
    run_dir = YOLO_RUNS_DIR / run_name
    best_path = run_dir / "weights" / "best.pt"
    if not best_path.exists():
        candidates = sorted(run_dir.glob("**/best.pt"))
        if not candidates:
            raise FileNotFoundError(f"Could not locate YOLO best checkpoint for {model_name} {loss_label}")
        best_path = candidates[-1]
    best_yolo = YOLO(str(best_path))
    print_gpu_status(f"{model_name} {loss_label} before YOLO eval")
    val_metrics = evaluate_yolo_model(best_yolo, val_df, timed=False)
    test_metrics = evaluate_yolo_model(best_yolo, test_df, timed=True)
    print_gpu_status(f"{model_name} {loss_label} after YOLO eval")
    best_epoch, best_val_top1 = get_yolo_best_info(run_dir)
    inf_time = max(test_metrics["elapsed"], 1e-9)
    params_m = count_params(best_yolo.model)
    cm_path = PLOT_DIR / f"confusion_matrix_{run_name}.png"
    save_confusion_matrix(test_metrics["labels"], test_metrics["preds"], f"{model_name} | {loss_label}", cm_path)
    history = read_yolo_history(run_dir, model_name, loss_name)
    result = {
        "Model": model_name,
        "Backend": "yolo",
        "Backend Name": weights_name,
        "Loss": loss_label,
        "Loss Key": loss_name,
        "Loss Paper": LOSS_PAPERS[loss_name],
        "Parameters (M)": round(params_m, 2),
        "Training Time (s)": round(train_time, 1),
        "Best Epoch": best_epoch,
        "Val F1-Score": round(val_metrics["macro_f1"], 4),
        "Best Val Loss": np.nan,
        "Best Val Top1": round(best_val_top1, 4) if not pd.isna(best_val_top1) else np.nan,
        "Test Accuracy": round(test_metrics["accuracy"], 4),
        "Test Precision": round(test_metrics["precision"], 4),
        "Test Recall": round(test_metrics["recall"], 4),
        "Test F1-Score": round(test_metrics["macro_f1"], 4),
        "Cohen Kappa": round(test_metrics["cohen_kappa"], 4),
        "Inference Time (s)": round(inf_time, 2),
        "FPS": round(len(test_df) / inf_time, 1),
        "Latency (ms)": round((inf_time / len(test_df)) * 1000, 2),
        "Checkpoint Path": str(best_path),
        "Confusion Matrix Path": str(cm_path),
        "Export Format": "",
        "Export Path": "",
        "Export Note": "",
    }
    del yolo, best_yolo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return result, history


Prepared YOLO classification dataset at /home/drnguyenvinh/notebooks/loss_ablation_yolo26m_original_pipeline_outputs/yolo_dataset


## 8. Run YOLOv26m-cls loss ablation


In [10]:
TARGET_RUNS = []
for model_name in YOLO_MODELS:
    for loss_name in LOSS_RUNS:
        TARGET_RUNS.append({"Model": model_name, "Backend": "yolo", "Loss": loss_name})

comparison_results = []
all_histories = []

for spec in TARGET_RUNS:
    model_name = spec["Model"]
    loss_name = spec["Loss"]
    backend = spec["Backend"]
    try:
        result, history = train_yolo_model(model_name, loss_name)
        comparison_results.append(result)
        all_histories.extend(history)
        print("Recorded result:")
        display(pd.DataFrame([result]))
    except Exception as exc:
        print(f"ERROR while running {model_name} | {LOSS_LABELS[loss_name]}: {type(exc).__name__}: {exc}")
        comparison_results.append(
            {
                "Model": model_name,
                "Backend": backend,
                "Backend Name": "",
                "Loss": LOSS_LABELS[loss_name],
                "Loss Key": loss_name,
                "Loss Paper": LOSS_PAPERS[loss_name],
                "Parameters (M)": np.nan,
                "Training Time (s)": np.nan,
                "Best Epoch": np.nan,
                "Val F1-Score": np.nan,
                "Best Val Loss": np.nan,
                "Best Val Top1": np.nan,
                "Test Accuracy": np.nan,
                "Test Precision": np.nan,
                "Test Recall": np.nan,
                "Test F1-Score": np.nan,
                "Cohen Kappa": np.nan,
                "Inference Time (s)": np.nan,
                "FPS": np.nan,
                "Latency (ms)": np.nan,
                "Checkpoint Path": "",
                "Confusion Matrix Path": "",
                "Export Format": "not_run",
                "Export Path": "",
                "Export Note": f"Run failed: {type(exc).__name__}: {exc}",
            }
        )
    partial_df = pd.DataFrame(comparison_results)
    partial_df.to_csv(REPORT_DIR / "yolo26m_loss_ablation_partial.csv", index=False)
    partial_df.to_json(REPORT_DIR / "yolo26m_loss_ablation_partial.json", orient="records", indent=2)
    if all_histories:
        pd.DataFrame(all_histories).to_csv(REPORT_DIR / "yolo26m_loss_ablation_history_partial.csv", index=False)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()



Training YOLO classifier: yolo26m-cls | Baseline CE
[GPU] yolo26m-cls Baseline CE before YOLO train: torch CUDA memory allocated/reserved/max: 0.0/0.0/0.0 MB | nvidia-smi util=29% mem=1241/24564 MB power=13.09 W temp=53 C
New https://pypi.org/project/ultralytics/8.4.53 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.13.2 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24072MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/drnguyenvinh/notebooks/loss_ablation_yolo26m_original_pipeline_outputs/yolo_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fract

<Figure size 650x550 with 2 Axes>

Recorded result:


,Model,Backend,Backend Name,Loss,Loss Key,Loss Paper,Parameters (M),Training Time (s),Best Epoch,Val F1-Score,...,Test F1-Score,Cohen Kappa,Inference Time (s),FPS,Latency (ms),Checkpoint Path,Confusion Matrix Path,Export Format,Export Path,Export Note
0,yolo26m-cls,yolo,yolo26m-cls.pt,Baseline CE,baseline_ce,Default CrossEntropyLoss,10.35,232.9,23,0.8258,...,0.8504,0.8033,12.21,14.2,70.57,/home/drnguyenvinh/notebooks/loss_ablation_yol...,/home/drnguyenvinh/notebooks/loss_ablation_yol...,,,



Training YOLO classifier: yolo26m-cls | ASL
[GPU] yolo26m-cls ASL before YOLO train: torch CUDA memory allocated/reserved/max: 65.0/68.0/4616.3 MB | nvidia-smi util=5% mem=1811/24564 MB power=182.92 W temp=53 C
New https://pypi.org/project/ultralytics/8.4.53 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.13.2 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24072MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/drnguyenvinh/notebooks/loss_ablation_yolo26m_original_pipeline_outputs/yolo_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fr

...

,Model,Backend,Backend Name,Loss,Loss Key,Loss Paper,Parameters (M),Training Time (s),Best Epoch,Val F1-Score,...,Test F1-Score,Cohen Kappa,Inference Time (s),FPS,Latency (ms),Checkpoint Path,Confusion Matrix Path,Export Format,Export Path,Export Note
0,yolo26m-cls,yolo,yolo26m-cls.pt,ASL,asl,Asymmetric Loss For Multi-Label Classification,10.35,197.1,26,0.8269,...,0.8997,0.8665,11.57,15.0,66.86,/home/drnguyenvinh/notebooks/loss_ablation_yol...,/home/drnguyenvinh/notebooks/loss_ablation_yol...,,,



Training YOLO classifier: yolo26m-cls | LDAM
[GPU] yolo26m-cls LDAM before YOLO train: torch CUDA memory allocated/reserved/max: 65.0/68.0/4616.3 MB | nvidia-smi util=5% mem=1826/24564 MB power=68.34 W temp=51 C
New https://pypi.org/project/ultralytics/8.4.53 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.13.2 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24072MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/drnguyenvinh/notebooks/loss_ablation_yolo26m_original_pipeline_outputs/yolo_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, f

<Figure size 650x550 with 2 Axes>

Recorded result:


,Model,Backend,Backend Name,Loss,Loss Key,Loss Paper,Parameters (M),Training Time (s),Best Epoch,Val F1-Score,...,Test F1-Score,Cohen Kappa,Inference Time (s),FPS,Latency (ms),Checkpoint Path,Confusion Matrix Path,Export Format,Export Path,Export Note
0,yolo26m-cls,yolo,yolo26m-cls.pt,LDAM,ldam,Learning Imbalanced Datasets with Label-Distri...,10.35,201.7,25,0.8355,...,0.8524,0.8096,11.54,15.0,66.73,/home/drnguyenvinh/notebooks/loss_ablation_yol...,/home/drnguyenvinh/notebooks/loss_ablation_yol...,,,


## 9. Final YOLO metrics table


In [11]:
df_summary = pd.DataFrame(comparison_results)
metric_columns = [
    "Model",
    "Backend",
    "Loss",
    "Parameters (M)",
    "Training Time (s)",
    "Best Epoch",
    "Best Val Top1",
    "Val F1-Score",
    "Best Val Loss",
    "Test Accuracy",
    "Test Precision",
    "Test Recall",
    "Test F1-Score",
    "Cohen Kappa",
    "Inference Time (s)",
    "FPS",
    "Latency (ms)",
]

if not df_summary.empty:
    for col in metric_columns:
        if col not in df_summary.columns:
            df_summary[col] = np.nan
    df_summary = df_summary.sort_values(by=["Model", "Loss Key"], ascending=[True, True], na_position="last").reset_index(drop=True)
    print("\n" + "=" * 100)
    print("YOLOV26M-CLS LOSS ABLATION PERFORMANCE SUMMARY")
    print("=" * 100)
    display(df_summary[metric_columns])
    summary_csv = REPORT_DIR / "yolo26m_loss_ablation_summary.csv"
    summary_json = REPORT_DIR / "yolo26m_loss_ablation_summary.json"
    metrics_csv = REPORT_DIR / "yolo26m_loss_ablation_metrics_table.csv"
    df_summary.to_csv(summary_csv, index=False)
    df_summary.to_json(summary_json, orient="records", indent=2)
    df_summary[metric_columns].to_csv(metrics_csv, index=False)
    print(f"Saved summary CSV: {summary_csv}")
    print(f"Saved summary JSON: {summary_json}")
    print(f"Saved metrics-only CSV: {metrics_csv}")
else:
    print("No model results were produced.")



YOLOV26M-CLS LOSS ABLATION PERFORMANCE SUMMARY


,Model,Backend,Loss,Parameters (M),Training Time (s),Best Epoch,Best Val Top1,Val F1-Score,Best Val Loss,Test Accuracy,Test Precision,Test Recall,Test F1-Score,Cohen Kappa,Inference Time (s),FPS,Latency (ms)
0,yolo26m-cls,yolo,ASL,10.35,197.1,26,0.8372,0.8269,NaN,0.9017,0.8956,0.9141,0.8997,0.8665,11.57,15.0,66.86
1,yolo26m-cls,yolo,Baseline CE,10.35,232.9,23,0.8372,0.8258,NaN,0.8555,0.8499,0.8606,0.8504,0.8033,12.21,14.2,70.57
2,yolo26m-cls,yolo,LDAM,10.35,201.7,25,0.8430,0.8355,NaN,0.8613,0.8601,0.8523,0.8524,0.8096,11.54,15.0,66.73


Saved summary CSV: /home/drnguyenvinh/notebooks/loss_ablation_yolo26m_original_pipeline_outputs/reports/yolo26m_loss_ablation_summary.csv
Saved summary JSON: /home/drnguyenvinh/notebooks/loss_ablation_yolo26m_original_pipeline_outputs/reports/yolo26m_loss_ablation_summary.json
Saved metrics-only CSV: /home/drnguyenvinh/notebooks/loss_ablation_yolo26m_original_pipeline_outputs/reports/yolo26m_loss_ablation_metrics_table.csv


## 10. YOLO visualization


In [12]:
if not df_summary.empty:
    plot_metrics = ["Test Accuracy", "Test Precision", "Test Recall", "Test F1-Score", "Cohen Kappa", "Latency (ms)"]
    for metric in plot_metrics:
        plot_df = df_summary.dropna(subset=[metric]).copy()
        if plot_df.empty:
            continue
        pivot = plot_df.pivot(index="Model", columns="Loss", values=metric)
        ax = pivot.plot(kind="bar", figsize=(9, 5))
        ax.set_title(metric)
        ax.set_xlabel("Model")
        ax.set_ylabel(metric)
        ax.legend(title="Loss")
        plt.xticks(rotation=0)
        plt.tight_layout()
        out_path = PLOT_DIR / f"bar_yolo26m_{sanitize_name(metric)}.png"
        plt.savefig(out_path, dpi=180, bbox_inches="tight")
        plt.show()

if all_histories:
    history_df = pd.DataFrame(all_histories)
    history_csv = REPORT_DIR / "yolo26m_loss_ablation_history.csv"
    history_df.to_csv(history_csv, index=False)
    display(history_df.head())
    curve_candidates = [
        "train/loss",
        "metrics/accuracy_top1",
        "metrics/accuracy_top5",
        "lr/pg0",
    ]
    for metric in curve_candidates:
        if metric not in history_df.columns:
            continue
        fig, ax = plt.subplots(figsize=(9, 5))
        for (model, loss), group in history_df.groupby(["Model", "Loss"]):
            group = group.sort_values("Epoch")
            ax.plot(group["Epoch"], group[metric], marker="o", label=f"{model} | {loss}")
        ax.set_title(f"YOLO {metric}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric)
        ax.legend()
        plt.tight_layout()
        out_path = PLOT_DIR / f"curve_yolo26m_{sanitize_name(metric)}.png"
        plt.savefig(out_path, dpi=180, bbox_inches="tight")
        plt.show()
    print(f"Saved history CSV: {history_csv}")


<Figure size 900x500 with 1 Axes>

<Figure size 900x500 with 1 Axes>

<Figure size 900x500 with 1 Axes>

<Figure size 900x500 with 1 Axes>

<Figure size 900x500 with 1 Axes>

...

<Figure size 900x500 with 1 Axes>

<Figure size 900x500 with 1 Axes>

<Figure size 900x500 with 1 Axes>

<Figure size 900x500 with 1 Axes>

Saved history CSV: /home/drnguyenvinh/notebooks/loss_ablation_yolo26m_original_pipeline_outputs/reports/yolo26m_loss_ablation_history.csv


## 11. Terminal execution command

From the repository root on Linux/RTX4090:

```bash
source .venv/bin/activate
jupyter nbconvert --to notebook --execute run_yolo26m_cls_custom_loss_original_pipeline_rtx4090.ipynb --output run_yolo26m_cls_custom_loss_original_pipeline_rtx4090_executed.ipynb --ExecutePreprocessor.timeout=-1
```

Main outputs:

```bash
loss_ablation_yolo26m_original_pipeline_outputs/reports/yolo26m_loss_ablation_summary.csv
loss_ablation_yolo26m_original_pipeline_outputs/reports/yolo26m_loss_ablation_metrics_table.csv
loss_ablation_yolo26m_original_pipeline_outputs/plots
loss_ablation_yolo26m_original_pipeline_outputs/checkpoints
loss_ablation_yolo26m_original_pipeline_outputs/yolo_runs
```
